[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrew-l-miller/gwosc/blob/main/create_sfdbs/crea_sfdbs.ipynb)

In [2]:
from pathlib import Path
import os
import requests
import argparse

# os.chdir('/Users/andrewmiller/Desktop/China/gwosc/')
!wget -nc https://dcc.ligo.org/public/0192/T2400058/003/segsH1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt
!wget -nc https://dcc.ligo.org/public/0192/T2400058/003/segsL1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt
    
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    
if IN_COLAB:
#     !wget -nc https://github.com/andrew-l-miller/gwosc/blob/main/create_sfdbs/compiled_sfdb_codes.zip
    !git clone https://github.com/andrew-l-miller/gwosc.git
    !cd gwosc/create_sfdbs/
    !unzip compiled_sfdb_codes.zip
    

from download_all_data_from_run import fetch_strain_list,download_strain_file 
from make_ffl import make_ffl
from convert_sciseg_file import convert_sciseg_file

File 'segsH1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt' already there; not retrieving.

File 'segsL1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt' already there; not retrieving.



In [97]:
## Function to download gravitational-wave frame (.gwf) files

def download_gwf(channel='O3b_4KHZ_R1',detector='H1',save_dir='./data/',gps_start=None,gps_end=None):
    os.makedirs(save_dir, exist_ok=True)

    # fetch strain files
    strain_files = fetch_strain_list(channel, detector,gps_start,gps_end)
    print(f"Found {len(strain_files)} files")
    try:
        with open("filesdone.txt", "r") as fp:
            donelist = [f.strip() for f in fp.readlines()]
    except FileNotFoundError:
        donelist = []
    for afile in strain_files:
        if afile["url"] in donelist:
            print("already downloaded")
            continue
        if afile["format"] == "gwf":
            print(f"Downloading {afile['url']}")
            fname = download_strain_file(afile["url"], outdir=save_dir)
            # tseries = TimeSeries.read(fname, format="hdf5.gwosc")
            with open("filesdone.txt", "a") as fp:
                fp.write(f"{afile['url']}\n")
            # process tseries here

            

            
def make_input_file(ifo,ffl_fname,sciseg_fname, channel_name,input_file_name,subsamp_factor='1',nsamps='4194304',interlace_ffts='2',window_type='5'):
    # ifo: H1, L1
    # ffl_fname:'H1_O3a.ffl'; list of frames with paths to them; create with make_ffl.py
    # sciseg_fname: 'H1_O3a_sciseg_for_sfdb.txt' contains seg number, start time, end time, duration; create with convert_sciseg_file 
    # channel_name: 'H1:GWOSC-4KHZ_R1_STRAIN'; channel in GWOSC to use
    # subsamp_fact: downsampling factor [1 = no downsampling]; for 4 KHz frames, no downsampling needed
    # nsamps: number of samples per FFT; def = 4194304, which means TFFT = 4194304/4096 = 1024 s
    # interlace_ffts: 2 for interlacing by 50%, 1 for no interlacing; 0 for ??
    # window_type: 0=no,1=Hann,2=Hamm,3=MAP, 4=Blackmann flatcos; 5=flat top,cosine edge. Sugg. 5) 
    flag_sfdb = '2'
    fact_evf = '2'
    verb_lvl = '1'
    max_num_FFTs_total = '1000000' ## maximum number of FFTs to do total; if large, does full data set (be careful for small TFFT)
    subsamp_fact_ar_spec = '128' ## factor by which AR specrum is downsammpled w.r.t. FFTs
    veto_freq = '100' ## frequency of veto
    subsamp_fact_veto = '-1' ##always keep this
    max_num_ffts_per_file = '100' ## max number of FFTs per SFDB

    lines = [
        str(ifo),                   # 1) detector
        str(flag_sfdb),             # 2) 2 or 3
        str(fact_evf),              # 3) "2" in template (keep as-is)
        str(ffl_fname),             # 4) .ffl file
        str(sciseg_fname),          # 5) science segments
        str(channel_name),          # 6) channel
        str(subsamp_factor),        # 7) subsampling factor
        str(verb_lvl),              # 8) extra 1 line in template (keep as-is)
        str(nsamps),                # 9) FFT length in samples
        str(interlace_ffts),        # 10) overlap/interlace
        str(max_num_FFTs_total),    # 11) max total FFTs
        str(subsamp_fact_ar_spec),  # 12) header spectrum subsampling
        str(window_type),           # 13) window type
        str(veto_freq),             # 14) veto frequency
        str(subsamp_fact_veto),     # 15) always -1
        str(max_num_ffts_per_file), # 16) FFTs per SFDB file
    ]

    outpath = Path(input_file_name)
    outpath.write_text("\n".join(lines) + "\n")

In [98]:
obs_run = 'O4a'
channel = obs_run+'_4KHZ_R1'
ifo = 'H1'
save_dir = './data/'+obs_run+'/'+ifo+'/'
gps_start = 1369185055
gps_end = gps_start+10*4096
path_to_crea_sfdb = './Pss_Fr_vPUBLIC/pss/pss_sfdb/'



In [99]:
## Download frames
download_gwf(channel,ifo,save_dir,gps_start,gps_end);

Found 22 files
already downloaded
already downloaded
already downloaded
already downloaded
already downloaded
already downloaded
already downloaded
already downloaded
already downloaded
already downloaded
already downloaded


In [3]:

## Make correctly formatted science segment file for crea_sfdb.c

if obs_run == 'O4a':
    if ifo == 'H1':
        O4a_scisegs = 'segsH1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt'
    elif ifo == 'L1':
        O4a_scisegs = 'segsL1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt'

fname_scisegs_for_sfdbs = ifo+'_'+obs_run+'_sciseg_for_sfdb.txt'

if IN_COLAB:
    convert_sciseg_file('../../'+O4a_scisegs,path_to_crea_sfdb+fname_scisegs_for_sfdbs);
else:
     convert_sciseg_file(O4a_scisegs,path_to_crea_sfdb+fname_scisegs_for_sfdbs);


NameError: name 'obs_run' is not defined

In [101]:
## Make frame list
ffl_name = ifo+'_'+channel+'.ffl'
make_ffl(save_dir,path_to_crea_sfdb+ffl_name,absolute=True)

## Make input file
input_file_name = ifo+'_'+channel+"_input_file"
make_input_file(ifo,ffl_name,fname_scisegs_for_sfdbs, channel,path_to_crea_sfdb+'/'+input_file_name);

Wrote 11 entries to ./Pss_Fr_vPUBLIC/pss/pss_sfdb/H1_O4a_4KHZ_R1.ffl (prefix='H-H1')


In [7]:
## Change directories to where we will run crea_sfdb
os.chdir(path_to_crea_sfdb)

In [ ]:
## Create the SFDBs
!bash -c "./crea_sfdb.out < {input_file_name}"

In [95]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])? y


bash: test_infile: No such file or directory
